# 02 - Huấn luyện các thực nghiệm

E0 SimpleCNN, E1 ResNet-18 frozen, E2 ResNet-18 fine-tuning và E3 fine-tuning + augmentation. Chọn checkpoint theo validation Macro-F1.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_dataloaders, seed_everything
from src.model import SimpleCNN, build_resnet18
from src.train import ExperimentLogger, fit
SEED, BATCH_SIZE, EPOCHS = 42, 32, 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
NUM_CLASSES = metadata['label'].nunique()
print('Device:', DEVICE, 'classes:', NUM_CLASSES)

In [ ]:
experiments = [
    ('E0_simplecnn', lambda: SimpleCNN(NUM_CLASSES), False, 1e-3),
    ('E1_resnet_frozen', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=True), False, 1e-3),
    ('E2_resnet_finetune', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=False), False, 1e-4),
    ('E3_resnet_finetune_aug', lambda: build_resnet18(NUM_CLASSES, freeze_backbone=False), True, 1e-4),
]
PROJECT_ROOT.joinpath('outputs').mkdir(exist_ok=True)
RUN_SEEDS = [42, 43, 44, 45, 46]  # 4 experiments x 5 seeds = 20 auditable runs
logger = ExperimentLogger(PROJECT_ROOT / 'outputs' / 'experiment_log.jsonl')
histories = {}
models = {}
for name, factory, augmentation, lr in experiments:
    for run_seed in RUN_SEEDS:
        run_id = f'{name}_seed{run_seed}'
        print(f'\n=== {run_id} ===')
        seed_everything(run_seed)
        loaders = make_dataloaders(metadata, BATCH_SIZE, augmentation=augmentation)
        model, history = fit(model=factory(), loaders=loaders, epochs=EPOCHS, device=DEVICE,
                             learning_rate=lr, checkpoint_path=PROJECT_ROOT / 'checkpoints' / f'{run_id}.pt',
                             logger=logger, run_id=run_id, experiment_name=name)
        histories[run_id] = pd.DataFrame(history)
        models[run_id] = model.cpu()
        histories[run_id].to_csv(PROJECT_ROOT / 'outputs' / f'{run_id}_history.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
PROJECT_ROOT.joinpath('outputs').mkdir(exist_ok=True)
for name, history in histories.items():
    plt.plot(history['epoch'], history['val_macro_f1'], label=name)
plt.xlabel('Epoch'); plt.ylabel('Validation Macro-F1'); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / 'outputs' / 'validation_f1_comparison.png', dpi=160); plt.show()